# 08.01 LangSmith Quickstart

LangSmith에 첫 트레이스를 보내고 UI에서 확인하는 것까지가 이 노트북의 범위다.
**환경변수 세 줄 + LangChain 코드 변경 0줄**이 전부다.

## 학습 목표

- LangSmith API 키 발급 후 `.env`에 주입한다
- `LANGSMITH_TRACING=true` 설정만으로 기존 에이전트 실행이 자동 트레이싱되는 것을 확인한다
- 트레이스에 `run_name`, `tags`, `metadata` 를 부착해 검색 가능하게 만든다
- UI에서 한 트레이스의 LLM/tool call 트리, latency, token 사용량, cost를 읽는다

## 무엇을 얻나

- 에이전트가 "왜 이런 답을 냈는지" 재현 가능한 실행 이력
- 다음 단계(평가·프롬프트 튜닝)의 원천 데이터

## 8.01.1 API 키 · 환경변수

1. https://smith.langchain.com/ → `Settings → API Keys → Create API Key`
2. `.env` 에 세 줄 추가:

```dotenv
LANGSMITH_API_KEY=lsv2_pt_xxxxxxxx
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=langsmith-quickstart
```

> `LANGSMITH_PROJECT` 는 UI의 프로젝트 이름. 미설정 시 `default` 에 기록된다.

**온보딩 플로우 (최초 1회)**

![역할 선택](../assets/images/langsmith/01_quickstart/00_onboarding_step1_role.png)

*Technical 선택 → 개발자용 code-first 흐름.*

![LangSmith vs Fleet](../assets/images/langsmith/01_quickstart/01_onboarding_step2_mode.png)

*LangSmith(code-first) 와 Fleet(no-code) 이 분기된다. 이 노트북은 LangSmith.*

![Home 빈 상태](../assets/images/langsmith/01_quickstart/02_home_empty_state.png)

*온보딩 완료 후 Home — Tracing/Datasets/Prompts 모두 0/4.*

![Get Started with Tracing](../assets/images/langsmith/01_quickstart/03_get_started_tracing_dialog.png)

*Home 의 첫 번째 카드 → 4단계 퀵스타트 다이얼로그. `Generate API Key` 버튼이 1단계.*

![API Key 발급](../assets/images/langsmith/01_quickstart/04_api_key_generated_RAW.png)

*Generate API Key 클릭 시 `lsv2_pt_…` 키가 생성되고 3단계 env 블록에도 자동 주입된다. 키는 한 번만 표시되므로 즉시 `.env` 로 복사.*

In [ ]:
# !pip install -U langsmith langchain langchain-openai

from dotenv import load_dotenv
import os
load_dotenv(override=True)

assert os.environ.get("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY 미설정"
assert os.environ.get("LANGSMITH_TRACING") == "true", "LANGSMITH_TRACING=true 필요"
print("프로젝트:", os.environ.get("LANGSMITH_PROJECT", "default"))

## 8.01.2 첫 트레이스 — LangChain 에이전트

`create_agent` 는 자동 계측된다. 환경변수만 켜져 있으면 아래 실행이 LangSmith에 기록된다.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환한다."""
    return f"{city}: 맑음, 21°C"

agent = create_agent(
    model="openai:gpt-4.1",
    tools=[get_weather],
    system_prompt="당신은 친절한 날씨 봇입니다.",
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "서울 날씨 알려줘"}]
})
result["messages"][-1].content

실행 후 UI에서 확인:

1. https://smith.langchain.com/ → 좌측 `Projects` → `langsmith-quickstart`
2. 방금 실행이 한 행으로 보임 (run_name = `AgentExecutor` 기본값)
3. 클릭하면 LLM 호출 → tool 호출 → LLM 호출 트리와 각 단계의 input/output, latency, token이 보임

![Projects 리스트](../assets/images/langsmith/01_quickstart/05_projects_list.png)

*Tracing 메뉴 → 방금 에이전트가 찍은 `pr-jaunty-competitor-63` 프로젝트. Trace Count·P50/P99 Latency·Total Tokens·Total Cost 7일 집계가 자동으로 채워진다.*

![프로젝트 Run 리스트](../assets/images/langsmith/01_quickstart/06_project_runs_list.png)

*프로젝트 클릭 → 개별 run 테이블. Input / Output / Latency / Tokens / Cost / Tags / Metadata 한눈에. 탭: Runs · Threads · Evaluators · Automations · Insights.*

![Trace Tree](../assets/images/langsmith/01_quickstart/07_trace_tree_view.png)

*run 하나 클릭 → waterfall tree. `model → tools → model` 순서와 각 단계 토큰/지연이 시각적으로 재구성된다.*

## 8.01.3 run_name · tags · metadata

`invoke(..., config=...)` 로 트레이스에 검색·필터용 메타 정보를 부착한다.

In [ ]:
config = {
    "run_name": "weather-bot:seoul-request",   # UI에 표시되는 이름
    "tags": ["env:dev", "feature:weather"],     # 필터 태그
    "metadata": {                                # 임의 KV
        "user_id": "u_00123",
        "session_id": "s_abc",
        "app_version": "0.5.0",
    },
}

agent.invoke(
    {"messages": [{"role": "user", "content": "부산 날씨 알려줘"}]},
    config=config,
)

UI에서 `Filter → Tags contains env:dev` 또는 `Metadata.user_id = u_00123` 으로 필터링되는 것을 확인한다.

![Attributes 탭](../assets/images/langsmith/01_quickstart/08_trace_attributes.png)

*Attributes 탭 — 위에서 붙인 Tags(`env:dev`, `feature:weather`), Metadata(`user_id`, `session_id`, `app_version`…)가 그대로 보이고, LangSmith 환경변수와 Runtime(파이썬 버전, 라이브러리 버전 등)이 자동 캡처된다.*

## 8.01.4 순수 Python 함수 트레이싱 — `@traceable`

LangChain을 거치지 않는 유틸리티 함수도 `@traceable`로 트레이스에 편입할 수 있다.

In [ ]:
from langsmith import traceable

@traceable(run_type="tool", name="normalize_city")
def normalize_city(raw: str) -> str:
    mapping = {"서울특별시": "서울", "부산광역시": "부산"}
    return mapping.get(raw, raw)

@traceable(run_type="chain", name="weather-pipeline")
def weather_pipeline(raw_city: str) -> str:
    city = normalize_city(raw_city)
    result = agent.invoke({"messages": [{"role": "user", "content": f"{city} 날씨 알려줘"}]})
    return result["messages"][-1].content

weather_pipeline("서울특별시")

UI에서 `weather-pipeline` 이 루트, 아래 `normalize_city` + `AgentExecutor` 가 자식으로 묶인 하나의 트리로 보인다.

## 8.01.5 트레이스를 코드에서 다시 조회

`langsmith.Client` 로 UI 없이도 트레이스를 프로그램적으로 꺼낸다. 평가·회귀 테스트의 입력이 된다.

In [ ]:
from langsmith import Client
client = Client()

runs = list(client.list_runs(
    project_name=os.environ["LANGSMITH_PROJECT"],
    run_type="chain",
    limit=5,
))
for r in runs:
    duration = (r.end_time - r.start_time).total_seconds() if r.end_time else 0.0
    print(f"{r.start_time:%H:%M:%S}  {r.name:30s}  {r.total_tokens or 0:>5} tok  {duration:.2f}s")

## 8.01.6 비용 · 토큰 집계

UI의 프로젝트 페이지 우상단 `Analytics` 탭에서 프로젝트 단위 총 비용/토큰 그래프를 볼 수 있다.
모델별 단가는 LangSmith가 자동으로 계산해 `total_cost` 필드에 채운다 (커스텀 모델은 설정 가능).

## 8.01.7 API 키 전용 페이지

온보딩 이후 키를 재발급하거나 revoke 하려면 `Settings → Access and Security → API Keys`.

![Settings API Keys](../assets/images/langsmith/01_quickstart/09_settings_api_keys.png)

*Settings 좌측 nav: API Keys · Members · OAuth providers · Provider secrets · MCP servers · Model configurations · **Fleet webhooks** · Model pricing · Feedback tags · Resource tags · Shared URLs · Plans · Usage · Credits · Invoices. 테이블의 Key 컬럼은 자동으로 앞뒤만 표시되고 Last Used At 이 기록된다.*

## 체크리스트

- [ ] `.env` 에 `LANGSMITH_API_KEY`, `LANGSMITH_TRACING=true`, `LANGSMITH_PROJECT` 세 줄 설정
- [ ] UI에서 방금 실행한 에이전트의 LLM/tool call 트리 확인
- [ ] `run_name` + `tags` + `metadata` 부착 후 UI 필터 동작 확인
- [ ] `@traceable` 로 일반 함수 트레이스에 편입
- [ ] `Client().list_runs(...)` 로 프로그램적 조회

## 다음

- `02_tracing_agents.ipynb` — 복잡한 에이전트(LangGraph subgraph, Deep Agents subagent)의 트레이스 구조
- `03_datasets_and_evaluation.ipynb` — 이 트레이스들을 데이터셋으로 만들어 자동 평가

## 참고

- LangSmith 공식: https://docs.smith.langchain.com/
- `docs/langchain/30-observability.md`
- `docs/OBSERVABILITY.md`